## Configuration

Set these before running. `SNOMED_DIR` should point at the folder you'd normally mount from
Google Drive in Colab — i.e. the unzipped SNOMED CT International RF2 release, containing a
`Snapshot/Terminology/` subfolder.

In [1]:
SNOMED_DIR = "/Users/baturu/Desktop/k_anonymity/Code_Cleaner_Version/project/SnomedCT_InternationalRF2_PRODUCTION_20260501T120000Z"
INPUT_JSON_PATH = "train.json"
# Minimum NER confidence to keep an extracted entity
MIN_NER_SCORE = 0.7

# Minimum UMLS-linking score to keep a candidate concept; how many candidates to keep per entity
UMLS_LINKER_THRESHOLD = 0.7
MAX_UMLS_CANDIDATES = 3

# Fuzzy match score (0-100, rapidfuzz WRatio) required to accept a SNOMED fuzzy match.
# Lower = more matches but more false positives. Raise this if you start seeing bad matches.
FUZZY_SCORE_CUTOFF = 90

## Step 1 — Load models

In [2]:
import json, re, glob
from collections import defaultdict

import pandas as pd
import spacy
from scispacy.linking import EntityLinker
from transformers import pipeline
from rapidfuzz import fuzz, process

# scispaCy biomedical pipeline + UMLS entity linker
nlp = spacy.load("en_core_sci_sm")
nlp.add_pipe(
    "scispacy_linker",
    config={
        "resolve_abbreviations": True,
        "linker_name": "umls",
        "max_entities_per_mention": MAX_UMLS_CANDIDATES,
        "threshold": UMLS_LINKER_THRESHOLD,
    },
)
linker = nlp.get_pipe("scispacy_linker")
print("scispaCy pipeline:", nlp.pipe_names)

# Biomedical NER model
ner = pipeline(
    "token-classification",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="max",  # better merging of wordpieces
)


/Users/baturu/Desktop/k_anonymity/Code_Cleaner_Version/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/baturu/Desktop/k_anonymity/Code_Cleaner_Version/venv/lib/python3.10/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
/Users/baturu/Desktop/k_anonymity/Code_Cleaner_Version/venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Us

scispaCy pipeline: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner', 'scispacy_linker']


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 6031.57it/s]


## Step 2 — Load dialogue data

In [3]:
with open(INPUT_JSON_PATH, "r") as f:
    dialogue_data = json.load(f)

print(f"Total dialogues loaded: {len(dialogue_data)}")

Total dialogues loaded: 482


## Step 3 — Extract entities + link to UMLS

Same boundary-fixing (dangling connectors / orphaned adjectives) and negation filtering as the
original NER notebook, plus full UMLS candidate capture (not just the top hit) for `umls_all`.

In [4]:
DANGLING_CONNECTORS = {"of", "in", "with", "to", "on", "and", "or", "for", "the"}
ORPHANED_ADJECTIVES = {
    "dry", "tight", "sharp", "dull", "severe", "mild",
    "chronic", "acute", "throbbing", "stabbing", "burning",
    "heavy", "persistent", "productive", "swollen",
}
NEGATION_KEYWORDS = ["no ", "not ", "denies ", "without ", "free of ", "negative for ", "don't"]


def get_umls_candidates(spacy_doc, char_start, char_end):
    """UMLS candidates for the spaCy entity overlapping this transformer-NER span."""
    for spacy_ent in spacy_doc.ents:
        if spacy_ent.start_char <= char_end and spacy_ent.end_char >= char_start:
            if not spacy_ent._.kb_ents:
                continue
            candidates = []
            for cui, score in spacy_ent._.kb_ents:
                concept = linker.kb.cui_to_entity[cui]
                candidates.append({
                    "umls_cui": cui,
                    "umls_name": concept.canonical_name,
                    "umls_score": round(float(score), 3),
                    "umls_tuis": "|".join(concept.types),
                    "umls_aliases": "|".join(concept.aliases),
                })
            return candidates
    return []


rows = []

for i, item in enumerate(dialogue_data):
    for turn_idx, utterance in enumerate(item["utterances"]):
        if not utterance.strip():
            continue

        entities = ner(utterance)
        spacy_doc = nlp(utterance)

        for ent in entities:
            current_word = ent["word"].lower().strip()
            words_in_entity = current_word.split()

            if words_in_entity:
                last_word = words_in_entity[-1]
                if last_word in DANGLING_CONNECTORS or last_word in ORPHANED_ADJECTIVES:
                    text_after = utterance[ent["end"]:]
                    match = re.match(r"\s+([a-zA-Z0-9\-]+)", text_after)
                    if match:
                        ent["end"] += len(match.group(0))
                        ent["word"] = utterance[ent["start"]:ent["end"]]

            context_window = utterance[max(0, ent["start"] - 15): ent["start"]].lower()
            if any(neg in context_window for neg in NEGATION_KEYWORDS):
                continue

            umls_candidates = get_umls_candidates(spacy_doc, ent["start"], ent["end"])
            top = umls_candidates[0] if umls_candidates else {}

            rows.append({
                "dialogue_id":   i,
                "description":   item["description"],
                "turn_idx":      turn_idx,
                "original_term": ent["word"],
                "entity_type":   ent["entity_group"],
                "score":         round(ent["score"], 3),
                "char_start":    ent["start"],
                "char_end":      ent["end"],
                "turn_text":     utterance,
                "umls_cui":      top.get("umls_cui"),
                "umls_name":     top.get("umls_name"),
                "umls_score":    top.get("umls_score"),
                "umls_tuis":     top.get("umls_tuis"),
                "umls_aliases":  top.get("umls_aliases"),
                "umls_all":      json.dumps(umls_candidates) if umls_candidates else None,
            })

df = pd.DataFrame(rows)
print(f"Raw entities extracted: {len(df)}")


Raw entities extracted: 7451


In [6]:
KEEP_ENTITY_TYPES = {"Disease_disorder", "Medication", "Sign_symptom"}

df = df[df["entity_type"].isin(KEEP_ENTITY_TYPES)]
df = df[df["score"] >= MIN_NER_SCORE].reset_index(drop=True)

df.to_csv("entities.csv", index=False)
print(f"Kept {len(df)} entities across {df['dialogue_id'].nunique()} dialogues")
df.head(5)


Kept 2473 entities across 443 dialogues


,dialogue_id,description,turn_idx,original_term,entity_type,score,char_start,char_end,turn_text,umls_cui,umls_name,umls_score,umls_tuis,umls_aliases,umls_all
0,0,throat a bit sore and want to get a good imune...,0,sore,Sign_symptom,1.000,22,26,patient: throat a bit sore and want to get a g...,C0234233,Sore to touch,0.997,T184,Sore pain|Tenderness|tenderness|soreness|Sore ...,"[{""umls_cui"": ""C0234233"", ""umls_name"": ""Sore t..."
1,0,throat a bit sore and want to get a good imune...,1,pain,Sign_symptom,1.000,37,41,doctor: during this pandemic. throat pain can ...,C0242429,Sore Throat,0.985,T184,"Sore throat, NOS|Throat pain|Sore throat|Fauci...","[{""umls_cui"": ""C0242429"", ""umls_name"": ""Sore T..."
2,0,throat a bit sore and want to get a good imune...,1,sore,Sign_symptom,0.978,248,252,doctor: during this pandemic. throat pain can ...,C0031350,Pharyngitis,0.994,T047,inflamed throats|inflammation throat|throat in...,"[{""umls_cui"": ""C0031350"", ""umls_name"": ""Pharyn..."
3,0,throat a bit sore and want to get a good imune...,1,infections,Disease_disorder,0.956,323,333,doctor: during this pandemic. throat pain can ...,C5203670,COVID19 (disease),0.891,T047,2019 Novel Coronavirus Infection|SARS-CoV-2 Di...,"[{""umls_cui"": ""C5203670"", ""umls_name"": ""COVID1..."
4,0,throat a bit sore and want to get a good imune...,1,cough,Sign_symptom,1.000,389,394,doctor: during this pandemic. throat pain can ...,None,None,NaN,None,None,None


## Step 4 — Load SNOMED CT and build matching indexes

Builds: `term_to_snomed` (lowercase term → concept ID), `concept_to_term` (concept ID → FSN),
`parent_map` (concept ID → parent concept ID, from `isA` relationships), and an inverted-index
`block` of SNOMED terms by word, used to scope fuzzy matching.

In [7]:
import glob

SNOMED_DIR = "/Users/baturu/Desktop/k_anonymity/Code_Cleaner_Version/project/SnomedCT_InternationalRF2_PRODUCTION_20260501T120000Z"

files = glob.glob(
    f"{SNOMED_DIR}/**/*Relationship*.txt",
    recursive=True
)

for f in files:
    print(f)

/Users/baturu/Desktop/k_anonymity/Code_Cleaner_Version/project/SnomedCT_InternationalRF2_PRODUCTION_20260501T120000Z/Full/Terminology/sct2_StatedRelationship_Full_INT_20260501.txt
/Users/baturu/Desktop/k_anonymity/Code_Cleaner_Version/project/SnomedCT_InternationalRF2_PRODUCTION_20260501T120000Z/Full/Terminology/sct2_Relationship_Full_INT_20260501.txt
/Users/baturu/Desktop/k_anonymity/Code_Cleaner_Version/project/SnomedCT_InternationalRF2_PRODUCTION_20260501T120000Z/Full/Terminology/sct2_RelationshipConcreteValues_Full_INT_20260501.txt
/Users/baturu/Desktop/k_anonymity/Code_Cleaner_Version/project/SnomedCT_InternationalRF2_PRODUCTION_20260501T120000Z/Snapshot/Terminology/sct2_RelationshipConcreteValues_Snapshot_INT_20260501.txt


In [8]:
desc_files = glob.glob(f"{SNOMED_DIR}/**/sct2_Description_Snapshot*_INT_*.txt", recursive=True)
# rel_files = glob.glob(f"{SNOMED_DIR}/**/sct2_Relationship_Snapshot*_INT_*.txt", recursive=True)
rel_files = glob.glob(
    f"{SNOMED_DIR}/**/sct2_Relationship_Full_INT_*.txt",
    recursive=True
)
assert desc_files, "Could not find the SNOMED Description Snapshot file — check SNOMED_DIR"
assert rel_files, "Could not find the SNOMED Relationship Snapshot file — check SNOMED_DIR"


desc_df = pd.read_csv(desc_files[0], sep="\t", low_memory=False)
active_desc = desc_df[desc_df["active"] == 1].copy()

FSN_TYPE_ID = 900000000000003001
SYNONYM_TYPE_ID = 900000000000013009

fsn_rows = active_desc[active_desc["typeId"] == FSN_TYPE_ID]
concept_to_term = fsn_rows.set_index("conceptId")["term"].to_dict()

term_to_snomed = {}
for concept_id, term in concept_to_term.items():
    term_to_snomed[term.split(" (")[0].strip().lower()] = concept_id

syn_rows = active_desc[(active_desc["typeId"] == SYNONYM_TYPE_ID) & (active_desc["term"].notna())]
for _, row in syn_rows.iterrows():
    key = row["term"].strip().lower()
    term_to_snomed.setdefault(key, row["conceptId"])

snomed_vocab = list(term_to_snomed.keys())
print(f"SNOMED vocabulary size: {len(snomed_vocab):,} terms")

# "is a" relationships -> parent concept, for snomed_parent_id / snomed_parent_term
ISA_TYPE_ID = 116680003
rel_df = pd.read_csv(rel_files[0], sep="\t", low_memory=False)
isa_rel = rel_df[(rel_df["active"] == 1) & (rel_df["typeId"] == ISA_TYPE_ID)]
parent_map = isa_rel.groupby("sourceId")["destinationId"].first().to_dict()

# Word -> set of SNOMED terms containing that word, used to scope fuzzy matching
# to terms that already share a meaningful word (much faster + more relevant
# than fuzzy-comparing against the entire vocabulary).
STOPWORDS = {"of", "the", "and", "or", "in", "on", "to", "with", "for", "a", "an"}
inverted_index = defaultdict(set)
for term in snomed_vocab:
    for word in term.split():
        if word not in STOPWORDS and len(word) > 2:
            inverted_index[word].add(term)


def candidate_block(term):
    words = [w for w in term.split() if w not in STOPWORDS and len(w) > 2]
    block = set()
    for w in words:
        block |= inverted_index.get(w, set())
    return block


SNOMED vocabulary size: 810,257 terms


In [9]:
import re

def normalize(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s*-\s*", "-", text)       # "covid - 19" -> "covid-19"
    text = re.sub(r"\s+", " ", text)
    return text

desc_files = glob.glob(f"{SNOMED_DIR}/**/sct2_Description_Snapshot*_INT_*.txt", recursive=True)
# rel_files = glob.glob(f"{SNOMED_DIR}/**/sct2_Relationship_Snapshot*_INT_*.txt", recursive=True)
rel_files = glob.glob(
    f"{SNOMED_DIR}/**/sct2_Relationship_Full_INT_*.txt",
    recursive=True
)
assert desc_files, "Could not find the SNOMED Description Snapshot file — check SNOMED_DIR"
assert rel_files, "Could not find the SNOMED Relationship Snapshot file — check SNOMED_DIR"

desc_df = pd.read_csv(desc_files[0], sep="\t", low_memory=False)
active_desc = desc_df[desc_df["active"] == 1].copy()

FSN_TYPE_ID = 900000000000003001
SYNONYM_TYPE_ID = 900000000000013009

fsn_rows = active_desc[active_desc["typeId"] == FSN_TYPE_ID]
concept_to_term = fsn_rows.set_index("conceptId")["term"].to_dict()

term_to_snomed = {}
for concept_id, term in concept_to_term.items():
    term_to_snomed[normalize(term.split(" (")[0])] = concept_id

syn_rows = active_desc[(active_desc["typeId"] == SYNONYM_TYPE_ID) & (active_desc["term"].notna())]
for _, row in syn_rows.iterrows():
    key = normalize(row["term"])
    term_to_snomed.setdefault(key, row["conceptId"])

snomed_vocab = list(term_to_snomed.keys())
print(f"SNOMED vocabulary size: {len(snomed_vocab):,} terms")

ISA_TYPE_ID = 116680003
rel_df = pd.read_csv(rel_files[0], sep="\t", low_memory=False)
isa_rel = rel_df[(rel_df["active"] == 1) & (rel_df["typeId"] == ISA_TYPE_ID)]
parent_map = isa_rel.groupby("sourceId")["destinationId"].first().to_dict()

STOPWORDS = {"of", "the", "and", "or", "in", "on", "to", "with", "for", "a", "an"}
inverted_index = defaultdict(set)
for term in snomed_vocab:
    for word in term.split():
        if word not in STOPWORDS and len(word) > 2:
            inverted_index[word].add(term)


def candidate_block(term):
    words = [w for w in term.split() if w not in STOPWORDS and len(w) > 2]
    block = set()
    for w in words:
        block |= inverted_index.get(w, set())
    return block

SNOMED vocabulary size: 810,133 terms


## Step 5 — Match entities to SNOMED CT

4 tiers, in order: exact UMLS name → exact UMLS alias → exact raw term → blocked fuzzy match
(tried against the UMLS name, then the raw term).

### v1

In [30]:
STOPWORDS = {
    "a", "an", "the", "of", "in", "on", "for", "to", "and", "or",
    "with", "without", "history", "hx", "patient", "pt"
}

def find_snomed(umls_name, umls_aliases, original_term):
    # Tier 1: exact match on the canonical UMLS name
    if pd.notna(umls_name):
        key = str(umls_name).strip().lower()
        if key in term_to_snomed:
            sid = term_to_snomed[key]
            return sid, concept_to_term.get(sid), "umls_name_exact"

    # Tier 2: exact match on any UMLS alias
    if pd.notna(umls_aliases):
        for alias in str(umls_aliases).split("|"):
            key = alias.strip().lower()
            if key and key in term_to_snomed:
                sid = term_to_snomed[key]
                return sid, concept_to_term.get(sid), "alias_exact"

    # Tier 3: exact match on the raw NER term
    if pd.notna(original_term):
        key = str(original_term).strip().lower()
        if key in term_to_snomed:
            sid = term_to_snomed[key]
            return sid, concept_to_term.get(sid), "original_exact"

    # Tier 4: blocked fuzzy match, token-order independent
    # Tier 4: blocked fuzzy match, with extra safety checks
    for candidate_text, label in ((umls_name, "umls_name_fuzzy"), (original_term, "original_fuzzy")):
        if pd.isna(candidate_text):
            continue

        clean = str(candidate_text).strip().lower()
        if not clean:
            continue

        block = candidate_block(clean)
        if not block:
            continue

        result = process.extractOne(
            clean,
            block,
            scorer=fuzz.WRatio,
            score_cutoff=FUZZY_SCORE_CUTOFF
        )

        if result:
            best_term, score, _ = result

            query_tokens = set(clean.split())
            match_tokens = set(best_term.split())

            shared_tokens = query_tokens & match_tokens
            shared_tokens = {
                tok for tok in shared_tokens
                if tok not in STOPWORDS and len(tok) > 2
            }

            # Safety rule:
            # if fuzzy match shares no meaningful medical word,
            # reject this match
            if not shared_tokens:
                continue

            sid = term_to_snomed[best_term]
            return sid, concept_to_term.get(sid), f"{label} ({best_term}, {score:.0f}%)"
    return None, None, None

snomed_ids, snomed_fsns, snomed_matches = [], [], []
for _, row in df.iterrows():
    sid, fsn, src = find_snomed(row.get("umls_name"), row.get("umls_aliases"), row.get("original_term"))
    snomed_ids.append(sid)
    snomed_fsns.append(fsn)
    snomed_matches.append(src)

df["snomed_id"] = snomed_ids
df["snomed_fsn"] = snomed_fsns
df["snomed_match"] = snomed_matches

matched_pct = df["snomed_id"].notna().mean() * 100
print(f"SNOMED match rate: {matched_pct:.1f}% ({df['snomed_id'].notna().sum()} / {len(df)})")
df["snomed_match"].value_counts(dropna=False)


SNOMED match rate: 91.3% (2258 / 2473)


snomed_match
umls_name_exact                                                                1300
alias_exact                                                                     477
original_exact                                                                  227
None                                                                            215
original_fuzzy (gu symptoms, 95%)                                                66
                                                                               ... 
original_fuzzy (undue concern and preoccupation with stressful events, 90%)       1
original_fuzzy (collection of coughed sputum, 90%)                                1
original_fuzzy (condition, 90%)                                                   1
umls_name_fuzzy (medication, 90%)                                                 1
original_fuzzy (exhausted platelets, 90%)                                         1
Name: count, Length: 123, dtype: int64

In [31]:
fuzzy_cases = df[
    df["snomed_match"].fillna("").str.contains("fuzzy", case=False)
][
    ["original_term", "entity_type", "score", "umls_name", "snomed_fsn", "snomed_match", "turn_text"]
]

fuzzy_cases.assign(
    turn_text=lambda x: x["turn_text"].str.slice(0, 200)
).head(50)

,original_term,entity_type,score,umls_name,snomed_fsn,snomed_match,turn_text
15,symptoms,Sign_symptom,0.968,Symptoms aspect,Symptoms: [genitourinary] or [urinary] (finding),"original_fuzzy (gu symptoms, 95%)",doctor: in brief: symptoms if you are infected...
16,symptoms,Sign_symptom,0.995,Symptoms aspect,Symptoms: [genitourinary] or [urinary] (finding),"original_fuzzy (gu symptoms, 95%)",doctor: in brief: symptoms if you are infected...
19,infective,Sign_symptom,1.000,Infectious Dose,Unit dose (qualifier value),"umls_name_fuzzy (dose, 90%)",doctor: in brief: symptoms if you are infected...
20,self,Sign_symptom,0.990,Self-Isolation,Ability to dry self (observable entity),"original_fuzzy (ability to dry self, 90%)",doctor: in brief: symptoms if you are infected...
21,isolation,Sign_symptom,0.999,Self-Isolation,Isolation of brachiocephalic trunk (disorder),original_fuzzy (isolation of innominate artery...,doctor: in brief: symptoms if you are infected...
22,symptoms,Sign_symptom,1.000,Symptoms aspect,Symptoms: [genitourinary] or [urinary] (finding),"original_fuzzy (gu symptoms, 95%)",doctor: in brief: symptoms if you are infected...
27,antibiotics,Medication,0.999,NaN,Adverse reaction to antineoplastic antibiotics...,original_fuzzy (adverse reaction to antineopla...,doctor: thanks for your question on healthcare...
46,symptoms,Sign_symptom,0.992,Symptoms aspect,Symptoms: [genitourinary] or [urinary] (finding),"original_fuzzy (gu symptoms, 95%)",doctor: symptoms. symptoms lasting for a week ...
56,strep,Disease_disorder,1.000,NaN,STREP BAC WITH IMUGEN II (product),"original_fuzzy (strep bac with imugen ii, 90%)",doctor: in brief: maybe. do video w/dr throat ...
58,corona,Disease_disorder,0.997,NaN,Structure of corona of penis (body structure),"original_fuzzy (structure of corona of penis, ...",patient: my 18 month old woke up from her nap ...


In [32]:
fuzzy_cases["entity_type"].value_counts()

entity_type
Sign_symptom        171
Disease_disorder     49
Medication           34
Name: count, dtype: int64

In [33]:
fuzzy_cases.groupby("entity_type").size()

entity_type
Disease_disorder     49
Medication           34
Sign_symptom        171
dtype: int64

### v2

In [34]:
STOPWORDS = {
    "a", "an", "the", "of", "in", "on", "for", "to", "and", "or",
    "with", "without", "history", "hx", "patient", "pt",
    "doctor", "dr", "hello", "thanks"
}

BLOCKLIST = {
    "symptom",
    "symptoms",
    "hurt",
    "pain",
    "self",
    "isolation",
    "help",
    "care",
    "well",
    "unwell"
}


def find_snomed(umls_name, umls_aliases, original_term, entity_type):
    # Tier 1: exact match on the canonical UMLS name
    if pd.notna(umls_name):
        key = str(umls_name).strip().lower()
        if key in term_to_snomed:
            sid = term_to_snomed[key]
            return sid, concept_to_term.get(sid), "umls_name_exact"

    # Tier 2: exact match on any UMLS alias
    if pd.notna(umls_aliases):
        for alias in str(umls_aliases).split("|"):
            key = alias.strip().lower()
            if key and key in term_to_snomed:
                sid = term_to_snomed[key]
                return sid, concept_to_term.get(sid), "alias_exact"

    # Tier 3: exact match on the raw NER term
    if pd.notna(original_term):
        key = str(original_term).strip().lower()
        if key in term_to_snomed:
            sid = term_to_snomed[key]
            return sid, concept_to_term.get(sid), "original_exact"

    # Important safety rule:
    # Sign_symptom terms are often too broad, so we do NOT fuzzy-match them.
    # They can still be matched by exact match above.
    if entity_type == "Sign_symptom":
        return None, None, None

    # Tier 4: blocked fuzzy match, with extra safety checks
    for candidate_text, label in (
        (umls_name, "umls_name_fuzzy"),
        (original_term, "original_fuzzy")
    ):
        if pd.isna(candidate_text):
            continue

        clean = str(candidate_text).strip().lower()
        if not clean:
            continue

        if clean in BLOCKLIST:
            continue

        block = candidate_block(clean)
        if not block:
            continue

        result = process.extractOne(
            clean,
            block,
            scorer=fuzz.WRatio,
            score_cutoff=FUZZY_SCORE_CUTOFF
        )

        if result:
            best_term, score, _ = result

            query_tokens = set(clean.split())
            match_tokens = set(best_term.split())

            shared_tokens = query_tokens & match_tokens
            shared_tokens = {
                tok for tok in shared_tokens
                if tok not in STOPWORDS and len(tok) > 2
            }

            if not shared_tokens:
                continue

            sid = term_to_snomed[best_term]
            return sid, concept_to_term.get(sid), f"{label} ({best_term}, {score:.0f}%)"

    return None, None, None

snomed_ids, snomed_fsns, snomed_matches = [], [], []

for _, row in df.iterrows():
    sid, fsn, src = find_snomed(
        row.get("umls_name"),
        row.get("umls_aliases"),
        row.get("original_term"),
        row.get("entity_type")
    )

    snomed_ids.append(sid)
    snomed_fsns.append(fsn)
    snomed_matches.append(src)

df["snomed_id"] = snomed_ids
df["snomed_fsn"] = snomed_fsns
df["snomed_match"] = snomed_matches

matched_pct = df["snomed_id"].notna().mean() * 100
print(f"SNOMED match rate: {matched_pct:.1f}% ({df['snomed_id'].notna().sum()} / {len(df)})")
df["snomed_match"].value_counts(dropna=False)

SNOMED match rate: 84.4% (2087 / 2473)


snomed_match
umls_name_exact                                                             1300
alias_exact                                                                  477
None                                                                         386
original_exact                                                               227
original_fuzzy (strep bac with imugen ii, 90%)                                10
umls_name_fuzzy (structure of corona of penis, 90%)                            7
umls_name_fuzzy (control, 90%)                                                 7
umls_name_fuzzy (hypoglycemic coma due to diabetes mellitus, 90%)              6
original_fuzzy (anti c, 90%)                                                   4
original_fuzzy (corn virus, 91%)                                               3
original_fuzzy (hpv dna detection, 90%)                                        3
original_fuzzy (maternal postpartum viral disease, 90%)                        2
original_fuzzy 

In [35]:
fuzzy_cases = df[
    df["snomed_match"].fillna("").str.contains("fuzzy", case=False)
][
    ["original_term", "entity_type", "score", "umls_name", "snomed_fsn", "snomed_match", "turn_text"]
]

fuzzy_cases.assign(
    turn_text=lambda x: x["turn_text"].str.slice(0, 200)
).head(50)

,original_term,entity_type,score,umls_name,snomed_fsn,snomed_match,turn_text
27,antibiotics,Medication,0.999,NaN,Adverse reaction to antineoplastic antibiotics...,original_fuzzy (adverse reaction to antineopla...,doctor: thanks for your question on healthcare...
56,strep,Disease_disorder,1.000,NaN,STREP BAC WITH IMUGEN II (product),"original_fuzzy (strep bac with imugen ii, 90%)",doctor: in brief: maybe. do video w/dr throat ...
58,corona,Disease_disorder,0.997,NaN,Structure of corona of penis (body structure),"original_fuzzy (structure of corona of penis, ...",patient: my 18 month old woke up from her nap ...
85,corona virus,Disease_disorder,1.000,NaN,Corn virus (organism),"original_fuzzy (corn virus, 91%)","doctor: stay home. right now stay home, keep r..."
114,corona,Disease_disorder,0.856,Corona,Structure of corona of penis (body structure),"umls_name_fuzzy (structure of corona of penis,...",doctor: unlikely covid. unlikely to be corona ...
127,diabetes,Disease_disorder,0.927,Diabetes,Hypoglycemic coma due to diabetes mellitus (di...,umls_name_fuzzy (hypoglycemic coma due to diab...,"doctor: hello, * the pneumonia does not give r..."
128,corona 19,Medication,0.996,Corona,Structure of corona of penis (body structure),"umls_name_fuzzy (structure of corona of penis,...",patient: will i be covered if i get corona 19 ...
129,medival aid,Medication,0.896,NaN,Aid (attribute),"original_fuzzy (aid, 90%)",patient: will i be covered if i get corona 19 ...
142,corona,Disease_disorder,0.748,Corona,Structure of corona of penis (body structure),"umls_name_fuzzy (structure of corona of penis,...",patient: i know hospitals filter air with hepa...
213,beta2,Medication,0.989,NaN,Interleukin-6 (substance),"original_fuzzy (interferon beta2, 90%)",patient: my son is 4 years old. at the age of ...


| original           | mapp                                        |
| ------------------- | ------------------------------------------ |
| corona              | Structure of corona of penis               |
| corona virus        | Corn virus                                 |
| corona 19           | Structure of corona of penis               |
| diabetes            | Hypoglycemic coma due to diabetes mellitus |
| probiotic           | Probiotic pillowcase                       |
| medications         | Drug (substance)                           |
| asthmatic condition | Condition (attribute)                      |
| tdap                | Unit (qualifier value)                     |
| ics                 | CONTROL (product)                          |


In [36]:
fuzzy_cases["entity_type"].value_counts()

entity_type
Disease_disorder    49
Medication          34
Name: count, dtype: int64

In [37]:
fuzzy_cases.groupby("entity_type").size()

entity_type
Disease_disorder    49
Medication          34
dtype: int64

### v3

In [53]:
STOPWORDS = {
    "a", "an", "the", "of", "in", "on", "for", "to", "and", "or",
    "with", "without", "history", "hx", "patient", "pt",
    "doctor", "dr", "hello", "thanks"
}

BLOCKLIST = {
    "symptom",
    "symptoms",
    "hurt",
    "pain",
    "self",
    "isolation",
    "help",
    "care",
    "well",
    "unwell"
}

COVID_TERMS = {
    "corona",
    "coronavirus",
    "corona virus",
    "covid",
    "covid19",
    "covid-19"
}

SHORT_TERM_BLOCKLIST = {
"ics",
"mbl",
"tdap",
}


def find_snomed(umls_name, umls_aliases, original_term, entity_type):

    # ==================================================
    # Tier 1: exact match on UMLS canonical name
    # ==================================================
    if pd.notna(umls_name):
        key = str(umls_name).strip().lower()

        if key in term_to_snomed:
            sid = term_to_snomed[key]
            return sid, concept_to_term.get(sid), "umls_name_exact"

    # ==================================================
    # Tier 2: exact match on UMLS aliases
    # ==================================================
    if pd.notna(umls_aliases):

        for alias in str(umls_aliases).split("|"):

            key = alias.strip().lower()

            if key and key in term_to_snomed:
                sid = term_to_snomed[key]

                return (
                    sid,
                    concept_to_term.get(sid),
                    "alias_exact"
                )

    # ==================================================
    # Tier 3: exact match on original NER term
    # ==================================================
    if pd.notna(original_term):

        key = str(original_term).strip().lower()

        if key in term_to_snomed:

            sid = term_to_snomed[key]

            return (
                sid,
                concept_to_term.get(sid),
                "original_exact"
            )

    # ==================================================
    # Do NOT fuzzy-match Sign_symptom
    # ==================================================
    if entity_type == "Sign_symptom":
        return None, None, None

    # ==================================================
    # Tier 4: fuzzy match
    # ==================================================
    for candidate_text, label in (
        (umls_name, "umls_name_fuzzy"),
        (original_term, "original_fuzzy")
    ):

        if pd.isna(candidate_text):
            continue

        clean = str(candidate_text).strip().lower()

        if not clean:
            continue

        # ----------------------------------------------
        # Generic terms
        # ----------------------------------------------
        if clean in BLOCKLIST:
            continue

        # ----------------------------------------------
        # COVID terms
        # ----------------------------------------------
        if clean in COVID_TERMS:
            continue

        # ----------------------------------------------
        # Problematic abbreviations
        # ----------------------------------------------
        if clean in SHORT_TERM_BLOCKLIST:
            continue

        # ----------------------------------------------
        # Candidate block
        # ----------------------------------------------
        block = candidate_block(clean)

        if not block:
            continue

        result = process.extractOne(
            clean,
            block,
            scorer=fuzz.WRatio,
            score_cutoff=FUZZY_SCORE_CUTOFF
        )

        if result:

            best_term, score, _ = result
            raw_term = None
            if pd.notna(original_term):
                raw_term = str(original_term).strip().lower()

            if raw_term in SHORT_TERM_BLOCKLIST:
                continue
            # ------------------------------------------
            # Require token overlap
            # ------------------------------------------
            query_tokens = set(clean.split())

            match_tokens = set(best_term.split())

            shared_tokens = query_tokens & match_tokens

            shared_tokens = {
                tok
                for tok in shared_tokens
                if tok not in STOPWORDS
                and len(tok) > 2
            }

            if not shared_tokens:
                continue

            sid = term_to_snomed[best_term]

            return (
                sid,
                concept_to_term.get(sid),
                f"{label} ({best_term}, {score:.0f}%)"
            )

    return None, None, None
snomed_ids, snomed_fsns, snomed_matches = [], [], []

for _, row in df.iterrows():
    sid, fsn, src = find_snomed(
        row.get("umls_name"),
        row.get("umls_aliases"),
        row.get("original_term"),
        row.get("entity_type")
    )

    snomed_ids.append(sid)
    snomed_fsns.append(fsn)
    snomed_matches.append(src)

df["snomed_id"] = snomed_ids
df["snomed_fsn"] = snomed_fsns
df["snomed_match"] = snomed_matches

matched_pct = df["snomed_id"].notna().mean() * 100
print(f"SNOMED match rate: {matched_pct:.1f}% ({df['snomed_id'].notna().sum()} / {len(df)})")
df["snomed_match"].value_counts(dropna=False)

SNOMED match rate: 83.5% (2066 / 2473)


snomed_match
umls_name_exact                                                             1300
alias_exact                                                                  477
None                                                                         407
original_exact                                                               227
original_fuzzy (strep bac with imugen ii, 90%)                                10
umls_name_fuzzy (hypoglycemic coma due to diabetes mellitus, 90%)              6
original_fuzzy (anti c, 90%)                                                   4
original_fuzzy (hpv dna detection, 90%)                                        3
original_fuzzy (virus, 90%)                                                    2
original_fuzzy (adverse reaction to antineoplastic antibiotics nos, 90%)       2
original_fuzzy (hiv viral load, 90%)                                           2
original_fuzzy (maternal postpartum viral disease, 90%)                        2
original_fuzzy 

#### check up

In [54]:
fuzzy_cases = df[
    df["snomed_match"].fillna("").str.contains("fuzzy", case=False)
][
    ["original_term", "entity_type", "score", "umls_name", "snomed_fsn", "snomed_match", "turn_text"]
]

fuzzy_cases.assign(
    turn_text=lambda x: x["turn_text"].str.slice(0, 200)
).head(50)

,original_term,entity_type,score,umls_name,snomed_fsn,snomed_match,turn_text
27,antibiotics,Medication,0.999,NaN,Adverse reaction to antineoplastic antibiotics...,original_fuzzy (adverse reaction to antineopla...,doctor: thanks for your question on healthcare...
56,strep,Disease_disorder,1.000,NaN,STREP BAC WITH IMUGEN II (product),"original_fuzzy (strep bac with imugen ii, 90%)",doctor: in brief: maybe. do video w/dr throat ...
127,diabetes,Disease_disorder,0.927,Diabetes,Hypoglycemic coma due to diabetes mellitus (di...,umls_name_fuzzy (hypoglycemic coma due to diab...,"doctor: hello, * the pneumonia does not give r..."
129,medival aid,Medication,0.896,NaN,Aid (attribute),"original_fuzzy (aid, 90%)",patient: will i be covered if i get corona 19 ...
213,beta2,Medication,0.989,NaN,Interleukin-6 (substance),"original_fuzzy (interferon beta2, 90%)",patient: my son is 4 years old. at the age of ...
221,decaf,Medication,0.998,NaN,Decaffeinated coffee (substance),"original_fuzzy (decaf coffee, 90%)",patient: i am a 76 year old female non smoker ...
244,probiotic,Medication,0.995,Probiotics,Probiotic pillowcase (physical object),"original_fuzzy (probiotic pillowcase, 90%)",patient: i was given doxycycline while in the ...
268,hiv,Disease_disorder,0.975,NaN,HIV viral load (procedure),"original_fuzzy (hiv viral load, 90%)",patient: why does hiv rna appear in blood earl...
342,diuretics,Medication,1.000,NaN,Harmful pattern of use of diuretic (disorder),"original_fuzzy (abuse of diuretics, 90%)",doctor: hello and welcome to ‘ask a doctor’ se...
460,ethyl,Medication,0.995,NaN,Ethyl violet stain (substance),"original_fuzzy (ethyl violet stain, 90%)",patient: are there additional risks associated...


In [55]:
fuzzy_cases[
    fuzzy_cases["original_term"].str.lower().isin(["ics", "mbl", "tdap"])
]

,original_term,entity_type,score,umls_name,snomed_fsn,snomed_match,turn_text


## Step 6 — Add SNOMED parent concept, assemble final table

In [40]:
def get_parent(sid):
    if pd.isna(sid):
        return None, None
    pid = parent_map.get(int(sid))
    return pid, (concept_to_term.get(pid) if pid else None)


parent_ids, parent_terms = [], []
for sid in df["snomed_id"]:
    pid, pterm = get_parent(sid)
    parent_ids.append(pid)
    parent_terms.append(pterm)

df["snomed_parent_id"] = parent_ids
df["snomed_parent_term"] = parent_terms

FINAL_COLUMNS = [
    "dialogue_id", "description", "turn_idx", "original_term", "entity_type",
    "score", "char_start", "char_end", "turn_text",
    "umls_cui", "umls_name", "umls_score", "umls_tuis", "umls_aliases", "umls_all",
    "snomed_id", "snomed_fsn", "snomed_match", "snomed_parent_id", "snomed_parent_term",
]
df = df[FINAL_COLUMNS]

df.to_csv("NER_fuzzy_entities_matched_to_snomed.csv", index=False)
print(f"Saved NER_fuzzy_entities_matched_to_snomed.csv with {len(df)} rows")
df.head(20)


Saved NER_fuzzy_entities_matched_to_snomed.csv with 2473 rows


,dialogue_id,description,turn_idx,original_term,entity_type,score,char_start,char_end,turn_text,umls_cui,umls_name,umls_score,umls_tuis,umls_aliases,umls_all,snomed_id,snomed_fsn,snomed_match,snomed_parent_id,snomed_parent_term
0,0,throat a bit sore and want to get a good imune...,0,sore,Sign_symptom,1.000,22,26,patient: throat a bit sore and want to get a g...,C0234233,Sore to touch,0.997,T184,Sore pain|Tenderness|tenderness|soreness|Sore ...,"[{""umls_cui"": ""C0234233"", ""umls_name"": ""Sore t...",247348008.0,Tenderness (finding),umls_name_exact,301370002.0,Finding of sensory dimension of pain (finding)
1,0,throat a bit sore and want to get a good imune...,1,pain,Sign_symptom,1.000,37,41,doctor: during this pandemic. throat pain can ...,C0242429,Sore Throat,0.985,T184,"Sore throat, NOS|Throat pain|Sore throat|Fauci...","[{""umls_cui"": ""C0242429"", ""umls_name"": ""Sore T...",267102003.0,Sore throat (finding),umls_name_exact,51388003.0,Pharyngeal pain (finding)
2,0,throat a bit sore and want to get a good imune...,1,sore,Sign_symptom,0.978,248,252,doctor: during this pandemic. throat pain can ...,C0031350,Pharyngitis,0.994,T047,inflamed throats|inflammation throat|throat in...,"[{""umls_cui"": ""C0031350"", ""umls_name"": ""Pharyn...",405737000.0,Pharyngitis (disorder),umls_name_exact,75860007.0,Disorder of pharynx (disorder)
3,0,throat a bit sore and want to get a good imune...,1,infections,Disease_disorder,0.956,323,333,doctor: during this pandemic. throat pain can ...,C5203670,COVID19 (disease),0.891,T047,2019 Novel Coronavirus Infection|SARS-CoV-2 Di...,"[{""umls_cui"": ""C5203670"", ""umls_name"": ""COVID1...",840539006.0,Disease caused by severe acute respiratory syn...,alias_exact,186747009.0,Coronavirus infection (disorder)
4,0,throat a bit sore and want to get a good imune...,1,cough,Sign_symptom,1.000,389,394,doctor: during this pandemic. throat pain can ...,NaN,NaN,NaN,NaN,NaN,NaN,49727002.0,Cough (finding),original_exact,301235001.0,Finding of cough (finding)
5,1,"hey there i have had cold ""symptoms"" for over ...",0,fever,Sign_symptom,0.999,82,87,"patient: hey there i have had cold ""symptoms"" ...",NaN,NaN,NaN,NaN,NaN,NaN,386661006.0,Fever (finding),original_exact,50177009.0,Body temperature above reference range (finding)
6,1,"hey there i have had cold ""symptoms"" for over ...",1,cough,Sign_symptom,1.000,193,198,doctor: yes. protection. it is not enough symp...,NaN,NaN,NaN,NaN,NaN,NaN,49727002.0,Cough (finding),original_exact,301235001.0,Finding of cough (finding)
7,1,"hey there i have had cold ""symptoms"" for over ...",1,shortness of breath,Sign_symptom,0.949,225,244,doctor: yes. protection. it is not enough symp...,C0013404,Dyspnea,0.965,T184,"Dyspnea, NOS|BREATHLESSNESS|Difficulty breathi...","[{""umls_cui"": ""C0013404"", ""umls_name"": ""Dyspne...",49233005.0,Dyspnea (finding),umls_name_exact,NaN,None
8,2,i have a tight and painful chest with a dry co...,0,coronavirus,Disease_disorder,0.999,108,119,patient: i have a tight and painful chest with...,C0206419,Genus: Coronavirus,0.977,T005,CORONAVIRUS|coronavirus|Coronaviruses|coronavi...,"[{""umls_cui"": ""C0206419"", ""umls_name"": ""Genus:...",243608008.0,Genus Coronavirus (organism),umls_name_exact,243607003.0,Family Coronaviridae (organism)
9,2,i have a tight and painful chest with a dry co...,1,fever,Sign_symptom,0.963,39,44,"doctor: possible. top symptoms include fever, ...",C0015967,Fever,0.962,T184,Pyrexia|Fever (finding)|increases temperature|...,"[{""umls_cui"": ""C0015967"", ""umls_name"": ""Fever""...",386661006.0,Fever (finding),umls_name_exact,50177009.0,Body temperature above reference range (finding)


## Tuning notes

- **More fuzzy matches, lower precision**: lower `FUZZY_SCORE_CUTOFF` (e.g. 75).
- **Fewer, more confident matches**: raise `FUZZY_SCORE_CUTOFF`, or require `MIN_NER_SCORE` higher.
- **Still missing matches you expect to see**: check `df["snomed_match"].value_counts()` above — if a
  term has no UMLS name/alias *and* no exact original-term hit, it's relying entirely on Tier 4.
  Things that help further: lemmatizing/singularizing terms before lookup, expanding common medical
  abbreviations before matching, or adding a second SNOMED extension file (e.g. a national edition)
  to `desc_files`/`rel_files` if you need locale-specific terms.
- `snomed_parent_id`/`snomed_parent_term` use the first active `isA` relationship found per concept;
  some SNOMED concepts have multiple parents — this notebook only surfaces one.

In [17]:
import pandas as pd

df = pd.read_csv("NER_fuzzy_entities_matched_to_snomed.csv")

unmatched = df[df["snomed_id"].isna()]
print(f"Unmatched: {len(unmatched)} / {len(df)} ({len(unmatched)/len(df):.1%})")

# Most common unmatched terms
print(unmatched["original_term"].value_counts().head(30))
#print(unmatched["original_term"].value_counts())

# Break down by entity type too
print(unmatched["entity_type"].value_counts())

Unmatched: 215 / 2473 (8.7%)
original_term
tylenol              9
pepcid               5
covid 19             4
zyrtec               4
tb                   4
sob                  3
probiotics           3
parangities          3
hurting              3
zantac               3
levaquin             3
covid - 19           2
asthama              2
advair               2
advil                2
mucinex              2
vte                  2
imodium              2
benadryl             2
emycin               2
nurofen              2
covid 19 symptoms    2
painkiller           2
azithromiacin        2
asthavent            1
convuls              1
cov19                1
cruddy               1
couldn               1
gi                   1
Name: count, dtype: int64
entity_type
Medication          133
Sign_symptom         50
Disease_disorder     32
Name: count, dtype: int64
